In [5]:
pip install mercantile

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [32]:
import pandas as pd

df1 = pd.read_parquet("2025-01-01_performance_mobile_tiles.parquet")
df2 = pd.read_parquet("2025-04-01_performance_mobile_tiles.parquet")
df3= pd.read_parquet("2025-07-01_performance_mobile_tiles.parquet")
df4 = pd.read_parquet("2025-10-01_performance_mobile_tiles.parquet")
df = pd.concat([df1, df2, df3, df4], ignore_index=True)

print(df.shape)

(13623459, 11)


In [33]:
df = df.drop_duplicates()

In [34]:
print(df.shape)

(13623459, 11)


In [35]:
import mercantile

def quadkey_to_latlon(quadkey):
    tile = mercantile.quadkey_to_tile(quadkey)
    bounds = mercantile.bounds(tile)
    
    # take center of tile
    lat = (bounds.north + bounds.south) / 2
    lon = (bounds.east + bounds.west) / 2
    
    return lat, lon

In [36]:
df[["lat", "lon"]] = df["quadkey"].apply(
    lambda x: pd.Series(quadkey_to_latlon(x))
)

In [37]:
df_ka = df[
    (df["lat"] >= 11.5) & (df["lat"] <= 18.5) &
    (df["lon"] >= 74.0) & (df["lon"] <= 78.5)
]

In [38]:
df_ka.shape

(156608, 13)

In [39]:
df_india = df[
    (df["lat"] >= 6.0) & (df["lat"] <= 37.0) &
    (df["lon"] >= 68.0) & (df["lon"] <= 97.0)
]

In [40]:
df_india = df_india.drop(df_ka.index)

In [41]:
df_india.shape

(1850034, 13)

In [42]:
df_india_sample = df_india.sample(n=400000, random_state=42)

In [43]:
df_final = pd.concat([df_ka, df_india_sample])

In [44]:
df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

In [45]:
df_final.to_csv("final_dataset.csv", index=False)

In [46]:
df_ka.to_csv("karnataka_data.csv", index=False)